In [2]:
import os
import time
import json
from pathlib import Path
import pandas as pd
import glob
import numpy as np
import pprint
import requests
from datetime import datetime

In [26]:
df_thursdays = pd.read_csv("thursday_vote_outcomes_council_win_or_lose.csv")
df_thursdays.head()

,Unnamed: 0.1,Unnamed: 0,vote_id,decision_id,start_date,method,outcome,attendees,votes_favor,votes_against,heading,master_doc,item_number,abstentions,Council win or loss
0,0,0,eli/dl/event/MTG-PL-2025-10-23-VOT-ITM-975331,eli/dl/event/MTG-PL-2025-10-23-DEC-180520,2025-10-23T13:08:06+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,571.0,220.0,341.0,Proposal to reject the Council position,NaN,NaN,NaN,Council win
1,19,19,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195775,2026-07-09T13:21:39+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,607.0,314.0,276.0,Proposal to reject the Council position,NaN,NaN,NaN,Council win
2,29,29,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195776,2026-07-09T13:22:14+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,605.0,117.0,422.0,Draft legislative act,NaN,NaN,NaN,Council win
3,24,24,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195777,2026-07-09T13:22:31+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,596.0,114.0,422.0,Draft legislative act,NaN,NaN,NaN,Council win
4,33,33,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195778,2026-07-09T13:22:50+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/ADOPTED,611.0,369.0,236.0,Draft legislative act,NaN,NaN,NaN,Council loss


In [27]:
#Doing a sanity check on votes that achieved absolute majority
df_thursdays[df_thursdays["votes_favor"] > 361]

,Unnamed: 0.1,Unnamed: 0,vote_id,decision_id,start_date,method,outcome,attendees,votes_favor,votes_against,heading,master_doc,item_number,abstentions,Council win or loss
4,33,33,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195778,2026-07-09T13:22:50+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/ADOPTED,611.0,369.0,236.0,Draft legislative act,NaN,NaN,NaN,Council loss
13,31,31,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195787,2026-07-09T13:25:49+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/ADOPTED,602.0,362.0,235.0,Draft legislative act,NaN,NaN,NaN,Council loss


In [28]:
#Sanity check for votes that achieved simple majority only
mask = (361 > df_thursdays['votes_favor']) & (df_thursdays['votes_favor'] > df_thursdays['votes_against'])
df_simple_maj = df_thursdays[mask].copy()

df_simple_maj

,Unnamed: 0.1,Unnamed: 0,vote_id,decision_id,start_date,method,outcome,attendees,votes_favor,votes_against,heading,master_doc,item_number,abstentions,Council win or loss
1,19,19,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195775,2026-07-09T13:21:39+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,607.0,314.0,276.0,Proposal to reject the Council position,NaN,NaN,NaN,Council win
9,14,14,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195783,2026-07-09T13:24:23+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,600.0,345.0,237.0,Draft legislative act,NaN,NaN,NaN,Council win
12,21,21,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195786,2026-07-09T13:25:26+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,603.0,346.0,254.0,Draft legislative act,NaN,NaN,NaN,Council win
14,30,30,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195788,2026-07-09T13:26:11+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,613.0,322.0,255.0,Draft legislative act,NaN,NaN,NaN,Council win
22,35,35,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195796,2026-07-09T13:28:06+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,614.0,353.0,254.0,Draft legislative act,NaN,NaN,NaN,Council win


In [29]:
df_weekdays = pd.read_csv("weekday_vote_results_council_wins_losses.csv")
df_weekdays.head()

,Unnamed: 0,vote_id,decision_id,master_doc,item_number,start_date,method,outcome,attendees,votes_favor,votes_against,abstentions,heading,Council win or loss
0,0,eli/dl/event/MTG-PL-2024-10-22-VOT-ITM-962879,eli/dl/event/MTG-PL-2024-10-22-DEC-170219,NaN,NaN,NaN,Auto-adopted,Approved,NaN,NaN,NaN,NaN,NaN,Council win
1,1,eli/dl/event/MTG-PL-2024-12-17-VOT-ITM-965069,eli/dl/event/MTG-PL-2024-12-17-DEC-171264,NaN,NaN,NaN,Auto-adopted,Approved,NaN,NaN,NaN,NaN,NaN,Council win
2,2,eli/dl/event/MTG-PL-2025-05-06-VOT-ITM-965509,eli/dl/event/MTG-PL-2025-05-06-DEC-176083,NaN,NaN,NaN,Auto-adopted,Approved,NaN,NaN,NaN,NaN,NaN,Council win
3,3,eli/dl/event/MTG-PL-2025-05-06-VOT-ITM-965500,eli/dl/event/MTG-PL-2025-05-06-DEC-176082,NaN,NaN,NaN,Auto-adopted,Approved,NaN,NaN,NaN,NaN,NaN,Council win
4,4,eli/dl/event/MTG-PL-2025-05-06-VOT-ITM-965505,eli/dl/event/MTG-PL-2025-05-06-DEC-176084,NaN,NaN,NaN,Auto-adopted,Approved,NaN,NaN,NaN,NaN,NaN,Council win


In [30]:
print(df_thursdays["Council win or loss"].value_counts(normalize=True, dropna=False) * 100)
print(df_weekdays["Council win or loss"].value_counts(normalize=True, dropna=False) * 100)

Council win or loss
Council win     93.333333
Council loss     4.444444
Unknown          2.222222
Name: proportion, dtype: float64
Council win or loss
Council win     92.452830
Council loss     5.660377
Unknown          1.886792
Name: proportion, dtype: float64


In [31]:
#Let's check what the "Unknown" outcomes actually were manually
df_thursdays[df_thursdays["Council win or loss"] == "Unknown"]
#Double checking this one shows that the vot itm 64 is missing from the official
#document recording the votes taken on that meeting

,Unnamed: 0.1,Unnamed: 0,vote_id,decision_id,start_date,method,outcome,attendees,votes_favor,votes_against,heading,master_doc,item_number,abstentions,Council win or loss
37,37,37,eli/dl/event/MTG-PL-2014-04-03-VOT-ITM-350882-64,NaN,NaN,Unknown,Unknown,NaN,NaN,NaN,NaN,eli/dl/doc/PV-7-2014-04-03-VOT,64.0,NaN,Unknown


In [32]:
df_weekdays[df_weekdays["Council win or loss"] == "Unknown"]
#double checking this one shows also that there is no record of an outcome

,Unnamed: 0,vote_id,decision_id,master_doc,item_number,start_date,method,outcome,attendees,votes_favor,votes_against,abstentions,heading,Council win or loss
28,29,eli/dl/event/MTG-PL-2015-03-10-VOT-ITM-409208-6,NaN,eli/dl/doc/PV-8-2015-03-10-VOT,6.0,NaN,Unknown,Unknown,NaN,NaN,NaN,NaN,NaN,Unknown


In [33]:
#To complete our dataframes we remove the rows with "Unknown" in "Counsil_win_or_loss"
df_thursdays = df_thursdays[df_thursdays["Council win or loss"] != "Unknown"]

In [34]:
df_thursdays = df_thursdays[df_thursdays["Council win or loss"] != "Unknown"]

In [35]:
df_weekdays = df_weekdays[df_weekdays["Council win or loss"] != "Unknown"]

In [36]:
print(df_thursdays["Council win or loss"].value_counts(normalize=True, dropna=False) * 100)
print(df_weekdays["Council win or loss"].value_counts(normalize=True, dropna=False) * 100)

Council win or loss
Council win     95.454545
Council loss     4.545455
Name: proportion, dtype: float64
Council win or loss
Council win     94.230769
Council loss     5.769231
Name: proportion, dtype: float64


In [37]:
df_thursdays.to_csv("thursday_votes_cleaned.csv")
df_weekdays.to_csv("weekday_votes_cleaned.csv")

In [38]:
df_thursdays["day_category"] = "Thursday"
df_weekdays["day_category"] = "Weekday"

In [39]:
#and one combined
df_combined = pd.concat([df_thursdays, df_weekdays], ignore_index=True)

In [40]:
df_combined = df_combined.drop(columns=["Unnamed: 0.1", "Unnamed: 0"])

In [41]:
df_combined.head()

,vote_id,decision_id,start_date,method,outcome,attendees,votes_favor,votes_against,heading,master_doc,item_number,abstentions,Council win or loss,day_category
0,eli/dl/event/MTG-PL-2025-10-23-VOT-ITM-975331,eli/dl/event/MTG-PL-2025-10-23-DEC-180520,2025-10-23T13:08:06+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,571.0,220.0,341.0,Proposal to reject the Council position,NaN,NaN,NaN,Council win,Thursday
1,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195775,2026-07-09T13:21:39+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,607.0,314.0,276.0,Proposal to reject the Council position,NaN,NaN,NaN,Council win,Thursday
2,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195776,2026-07-09T13:22:14+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,605.0,117.0,422.0,Draft legislative act,NaN,NaN,NaN,Council win,Thursday
3,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195777,2026-07-09T13:22:31+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,596.0,114.0,422.0,Draft legislative act,NaN,NaN,NaN,Council win,Thursday
4,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195778,2026-07-09T13:22:50+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/ADOPTED,611.0,369.0,236.0,Draft legislative act,NaN,NaN,NaN,Council loss,Thursday


In [42]:
df_combined.to_csv("combined_votes_cleaned.csv")

In [43]:
df_combined = pd.read_csv("combined_votes_cleaned.csv")

In [44]:
df_combined["day_category"].value_counts()

day_category
Weekday     52
Thursday    44
Name: count, dtype: int64

In [45]:
clean_dates = df_combined["vote_id"].str.replace("eli/dl/event/MTG-PL-", "").str[:10]

date_objs = pd.to_datetime(clean_dates, errors="coerce")
days = date_objs.dt.day_name()

not_thursdays = df_combined["day_category"] == "Weekday"
df_combined.loc[not_thursdays, "day_category"] = days[not_thursdays]

print(df_combined["day_category"].value_counts)



<bound method IndexOpsMixin.value_counts of 0      Thursday
1      Thursday
2      Thursday
3      Thursday
4      Thursday
        ...    
91    Wednesday
92    Wednesday
93      Tuesday
94      Tuesday
95    Wednesday
Name: day_category, Length: 96, dtype: str>


In [47]:
df_combined["day_category"].value_counts()

day_category
Thursday     44
Tuesday      32
Wednesday    20
Name: count, dtype: int64

In [48]:
df_combined.to_csv("combined_votes_cleaned.csv")